# Part 1: Data Acquisition
This code pulls the required compounds (transition metal halides) from Materials Project.<br>
We first input the MP API key (note: You will have to insert your own key yourself.  See readme.md for more).<br>
We follow this by creating a list of elements needed for the compounds (transition metals and halogens)--then use that list to pull data.<br>

In [30]:
#You have to do this patch to get MP pulls to work
import typing
import typing_extensions

# Manually 'backport' the missing attribute into the built-in typing module
if not hasattr(typing, "NotRequired"):
    typing.NotRequired = typing_extensions.NotRequired

if not hasattr(typing, "Required"):
    typing.Required = typing_extensions.Required

print("Successfully patched typing module for Python 3.10")

Successfully patched typing module for Python 3.10


In [31]:
# Cell 0 — Environment check
#Check the environment to make sure all packages load.  Copy week 5 due to use of rf model
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f'Python:      {sys.version.split()[0]}')
print(f'NumPy:       {np.__version__}')
print(f'pandas:      {pd.__version__}')
print(f'scikit-learn: already imported above')
print('\n\u2713 All imports successful.')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})

Python:      3.10.20
NumPy:       2.2.6
pandas:      2.3.3
scikit-learn: already imported above

✓ All imports successful.


In [32]:
#Import MP API Key
from dotenv import load_dotenv
import os
#make sure the figures directory exists so that we can join them together
output_dir = '../figures'
os.makedirs(output_dir, exist_ok=True)

load_dotenv()  # Reads .env file from repository root
API_KEY = os.getenv("MP_API_KEY")

if API_KEY:
    print(f"✅ MP API key loaded ({len(API_KEY)} characters)")
    print("   Key preview:", API_KEY[:4] + "*" * (len(API_KEY) - 8) + API_KEY[-4:])
else:
    print("❌ MP API key not found.")
    print("   Create a .env file in the repository root with:")
    print("   MP_API_KEY=your_actual_key_here")
    print("   Register for a free key at: https://next.materialsproject.org/api")

✅ MP API key loaded (32 characters)
   Key preview: AkMg************************UFWm


In [33]:
#Import transition metal halides from Materials Project
from mp_api.client import MPRester
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
#List all transition metals from periods 4-6 (leave out period 7 as these are short half life elements that don't exist in nature)
transition_metals = [
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg"
]
#Then list the halogens (with exception of period 7)
halogens = ["F", "Cl", "Br", "I", "At"]
all_results = []

#I used Gemini to help me with this code
print("Querying Materials Project...")
with MPRester(API_KEY) as mpr:
    for halogen in halogens:
        # Build chemical systems format: "TM-Halogen" (e.g., "Fe-Cl", "Ti-F")
        chemsys_list = [f"{tm}-{halogen}" for tm in transition_metals]
        
        # Query for both summary data and electronic structure fields
        # Need nsites to normalize by atom
        docs = mpr.summary.search(
            chemsys=chemsys_list,
            fields=[
                "material_id", 
                "formula_pretty", 
                "chemsys", 
                "symmetry",
                "total_magnetization",
                "nsites",
                "formation_energy_per_atom",
                "is_stable"
            ]
        )
        all_results.extend(docs)

print(f"Successfully retrieved {len(all_results)} materials.")

Querying Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/267 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/185 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/115 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/249 [00:00<?, ?it/s]

Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

Successfully retrieved 816 materials.
